In [3]:
raw = pd.read_excel("../data/census_2010_occ_soc.xls", header=None)
print(raw.head(15).to_string())

                                                        0                                                                                          1                 2                  3
0                                                     NaN                                                                           US Census Bureau               NaN                NaN
1                                                     NaN                                                                  2010 Occupation Code List               NaN                NaN
2                                                     NaN                                                                                        NaN               NaN                NaN
3                                                     NaN                                                              last updated: August 12, 2011               NaN                NaN
4                                                     NaN             

In [4]:
HEADER_ROW = 1   
census = pd.read_excel("../data/census_2010_occ_soc.xls", header=HEADER_ROW)
print(census.columns.tolist())
print(census.head(10).to_string())

['Unnamed: 0', '2010 Occupation Code List', 'Unnamed: 2', 'Unnamed: 3']
                                              Unnamed: 0                                                                  2010 Occupation Code List        Unnamed: 2         Unnamed: 3
0                                                    NaN                                                                                        NaN               NaN                NaN
1                                                    NaN                                                              last updated: August 12, 2011               NaN                NaN
2                                                    NaN                                                                Occupation 2010 Description  2010 Census Code      2010 SOC Code
3                                                    NaN                                                                                        NaN               NaN                NaN
4  

In [5]:
import pandas as pd, re

raw = pd.read_excel("../data/census_2010_occ_soc.xls", header=None)

census = raw.iloc[:, [2, 3]].copy()
census.columns = ["census_code", "soc2010"]
census = census.dropna()

census["census_code"] = census["census_code"].astype(str).str.strip()
census["soc2010"] = census["soc2010"].astype(str).str.strip()

# keep only real rows: 1-4 digit Census code, SOC like 11-1011 or 11-91XX
is_code = census["census_code"].str.fullmatch(r"\d{1,4}(\.0)?")
is_soc = census["soc2010"].str.fullmatch(r"\d{2}-[\dX]{4}")
census = census[is_code & is_soc].copy()

census["census_code"] = (census["census_code"].str.replace(r"\.0$", "", regex=True)
                                            .str.zfill(4))

print(len(census), "rows")
print(census.head(10))
print(census[census.soc2010.str.contains("X")].head(10))

538 rows
   census_code  soc2010
14        0010  11-1011
15        0020  11-1021
16        0030  11-1031
17        0040  11-2011
18        0050  11-2020
19        0060  11-2031
20        0100  11-3011
21        0110  11-3021
22        0120  11-3031
23        0135  11-3111
    census_code  soc2010
84         1020  15-113X
155        2025  21-109X
180        2550  25-90XX
254        3655  31-909X
275        3955  33-909X
297        4220  37-201X
312        4460  39-40XX
458        6940  47-50XX
468        7100  49-209X
483        7330  49-904X


In [ ]:
census = census.rename(columns={
    "2010 Census Code": "census_code",   # replace with the exact name printed
    "2010 SOC Code": "soc2010",          # replace with the exact name printed
})
census = census[["census_code", "soc2010"]].dropna()

In [6]:
dropped = raw.iloc[:, [2, 3]].dropna()
dropped.columns = ["census_code", "soc2010"]
print(dropped[~dropped.index.isin(census.index)].to_string())

          census_code            soc2010
4    2010 Census Code      2010 SOC Code
8           0010-3540  11-0000 - 29-0000
10          0010-0950  11-0000 - 13-0000
12          0010-0430            11-0000
45          0500-0950            13-0000
76          1000-1965  15-0000 - 19-0000
78          1000-1240            15-0000
97          1300-1560            17-0000
121         1600-1965            19-0000
147         2000-2960  21-0000 - 27-0000
149         2000-2060            21-0000
160         2100-2160            23-0000
168         2200-2550            25-0000
182         2600-2960            27-0000
204         3000-3540            29-0000
240         3600-4650  31-0000 - 39-0000
242         3600-3655            31-0000
256         3700-3955            33-0000
277         4000-4160            35-0000
293         4200-4250            37-0000
302         4300-4650            39-0000
325         4700-5940  41-0000 - 43-0000
327         4700-4965            41-0000
348         5000

In [7]:
dropped = raw.iloc[:, [2, 3]].dropna()
dropped.columns = ["census_code", "soc2010"]
dropped = dropped[~dropped.index.isin(census.index)]

# any single code (no range) that got dropped?
single = dropped[dropped["census_code"].astype(str).str.fullmatch(r"\d{1,4}(\.0)?")]
print(single.to_string())
print("kept:", len(census))

    census_code soc2010
635        9830    none
637        9920    none
kept: 538
